In [22]:
import os
import json
import pandas as pd
import numpy as np
import torch

In [23]:
with open("/home/bbc8731/NetMedGPT/data/parameters.json", 'r') as file:
    param = json.load(file)

data_dir = param['files']['data_dir']
queried_edge_types = ['indication']
queried_node_types = ['disease|drug']
random_state = 42

In [24]:
# Read edge_index and nodes
edge_index_df = pd.read_csv(os.path.join(data_dir, 'edges_ctg_reduced.csv'), sep= ',')
ctg = pd.read_csv(os.path.join(data_dir, "edges_clinical_trials.csv")) 

In [25]:
edges_one_side = ctg.loc[:,["disease", "drug", "xy"]]
edges_one_side.columns = ["x_index", "y_index", "xy"]
edges_one_side = edges_one_side.drop_duplicates()

In [26]:
def negative_samples(pos_edges, all_edges, all_drugs, random_state=123):
    np.random.seed(random_state)

    # Build a mapping of disease → known positive drugs
    disease_to_pos = pos_edges.groupby('x_index')['y_index'].apply(set).to_dict()
    
    # Set of all known disease-drug edges to avoid duplicates
    known_edges = set(zip(all_edges['x_index'], all_edges['y_index']))
    
    negative_edges = []
    for disease, pos_drugs in disease_to_pos.items():
        candidate_drugs = list(set(all_drugs) - pos_drugs)
            
        num_negatives_needed = len(pos_drugs)
        selected_negatives = set()
        np.random.shuffle(candidate_drugs)  # optional: randomize order
        for drug in candidate_drugs:
            if (disease, drug) not in known_edges:
                selected_negatives.add((disease, drug))
            if len(selected_negatives) == num_negatives_needed:
                break
        
        # Add selected negatives to final list
        negative_edges.extend(selected_negatives)
    
    return(negative_edges)

In [27]:
def to_edge_tensor(edge_list):
    if len(edge_list) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    else:
        return torch.tensor(edge_list, dtype=torch.long).T

In [28]:
# relations and their node sources
relation_node_types = {
    'protein_protein': ('NCBI', 'NCBI'),
    'drug_protein': ('DrugBank', 'NCBI'),
    'contraindication':('MONDO', 'DrugBank'),
    'indication': ('MONDO', 'DrugBank'),
    'off-label use': ('MONDO', 'DrugBank'),
    'drug_drug': ('DrugBank', 'DrugBank'),
    'phenotype_protein': ('HPO', 'NCBI'),
    'phenotype_phenotype': ('HPO', 'HPO'),
    'disease_phenotype_negative': ('MONDO', 'HPO'),
    'disease_phenotype_positive': ('MONDO', 'HPO'),
    'disease_protein': ('MONDO', 'NCBI'),
    'disease_disease': ('MONDO', 'MONDO'),
    'drug_effect': ('DrugBank', 'HPO'),
    'bioprocess_bioprocess': ('GO', 'GO'),
    'molfunc_molfunc': ('GO', 'GO'),
    'cellcomp_cellcomp': ('GO', 'GO'),
    'molfunc_protein': ('GO', 'NCBI'),
    'cellcomp_protein': ('GO', 'NCBI'),
    'bioprocess_protein': ('GO', 'NCBI'),
    'exposure_protein': ('CTD', 'NCBI'),
    'exposure_disease': ('CTD', 'MONDO'),
    'exposure_exposure': ('CTD', 'CTD'),
    'exposure_bioprocess': ('CTD', 'GO'),
    'exposure_molfunc': ('CTD', 'GO'),
    'exposure_cellcomp': ('CTD', 'GO'),
    'pathway_pathway': ('REACTOME', 'REACTOME'),
    'pathway_protein': ('REACTOME', 'NCBI'),
    'anatomy_anatomy': ('UBERON', 'UBERON'),
    'anatomy_protein_present': ('UBERON', 'NCBI'),
    'anatomy_protein_absent': ('UBERON', 'NCBI'),
}

In [29]:
queried_dict = {i:relation_node_types[i] for i in queried_edge_types}

## Train-val-test split of edge list of quiried types
test_data_edge_label_index = torch.tensor([], dtype=torch.long)
test_data_edge_label = torch.tensor([], dtype=torch.long)
edge_type_flag_test = torch.tensor([], dtype=torch.long)

for r, (i, (node_source1, node_source2)) in enumerate(queried_dict.items()):

    print(f"relation: {r}")
    
    edges_i = edge_index_df.loc[
            (edge_index_df['relation'] == i) & (edge_index_df['node_types'] == queried_node_types[r]), 
            ["x_index", "y_index", "xy"]].copy()

    #################################################################

    # all drugs that are connected to test nodes
    all_drug_ids = edges_i.loc[:,'y_index'].unique() # for neg_edges uses all drugs that are that relation
    
    negative_edges_test = negative_samples(edges_one_side, edge_index_df, all_drug_ids, random_state)
    positive_edges_test = list(zip(edges_one_side["x_index"], edges_one_side["y_index"]))
                                                            
    ## Convert to torch
    positive_edges_test = to_edge_tensor(positive_edges_test)
    negative_edges_test = to_edge_tensor(negative_edges_test)
    
    test_data_edge_label_index_i = torch.cat((positive_edges_test, negative_edges_test), dim = 1)
    test_data_edge_label_i = torch.cat((torch.ones(positive_edges_test.shape[1], dtype=torch.long), 
                                        torch.zeros(negative_edges_test.shape[1], dtype=torch.long)), dim = 0)

    edge_type_flag_test_i = (r+1) * torch.ones(test_data_edge_label_i.shape, dtype=torch.long)

    test_data_edge_label_index = torch.cat((test_data_edge_label_index, test_data_edge_label_index_i), dim = 1)

    test_data_edge_label = torch.cat((test_data_edge_label, test_data_edge_label_i), dim = 0)

    edge_type_flag_test = torch.cat((edge_type_flag_test, edge_type_flag_test_i), dim = 0)

    ####################################
    ### debugging
    # Confirm no overlap of transductive test nodes in train or val
    test_edges = set(edges_one_side["xy"])
    train_edges = set(edge_index_df["xy"])

    intersect_train_test = test_edges.intersection(train_edges)

    if intersect_train_test:
        print(f"❗ transductive violation: {len(intersect_train_test)} test diseases are in training!")
    else:
        print("✅ transductive isolation successful. No test diseases in train/val.")


relation: 0
✅ transductive isolation successful. No test diseases in train/val.


In [30]:
def add_relations_to_walk(walk, edge_to_rel):
    sequence = []
    for i in range(len(walk) - 1):
        src, dst = walk[i], walk[i + 1]
        rel = edge_to_rel.get((src, dst))
        sequence.extend([src, rel])
    sequence.append(walk[-1])
    return sequence

In [31]:
walks_df = edge_index_df.loc[:,["x_index", "y_index", "z_index"]]

# make a dict of edges and their relations
edge_to_rel = {
    (x, y): z
    for x, y, z in zip(walks_df['x_index'], walks_df['y_index'], walks_df['z_index'])
}

In [32]:
test_relation_row_all = torch.empty((1, 0), dtype=torch.long)
val_relation_row_all = torch.empty((1, 0), dtype=torch.long)
train_relation_row_all = torch.empty((1, 0), dtype=torch.long)

for i in range(len(queried_edge_types)):
    relation_token_i = edge_index_df.loc[edge_index_df['relation'] == queried_edge_types[i], 'z_index'].unique()
    test_data_edge_label_index_i = test_data_edge_label_index[:, edge_type_flag_test == i+1]
    num_test_edges_i = test_data_edge_label_index_i.shape[1]
    test_relation_row_i = torch.full((1, num_test_edges_i), relation_token_i.item(), dtype = test_data_edge_label_index_i.dtype)
    test_relation_row_all = torch.cat([test_relation_row_all, test_relation_row_i], dim=1)


test_data_edge_label_index_with_relation = torch.cat([test_data_edge_label_index, test_relation_row_all], dim=0)
test_data_edge_label_index_with_relation = test_data_edge_label_index_with_relation[[0,2,1],:]


In [33]:
# save data for each edge
torch.save(test_data_edge_label_index_with_relation, os.path.join(data_dir, f"test_data_edge_label_index_clinical_trials.pt"))
torch.save(test_data_edge_label, os.path.join(data_dir, f"test_data_edge_label_clinical_trials.pt"))
